# IEM Cologne Major 2026 Stage 1 因素雷达图

这个 notebook 读取 Stage 1 因素综合快照，并为 16 支队伍生成中文雷达图。维度来自 `cs2_match_prediction_framework.md` 中对胜负有价值的因素，但只给当前数据能支撑的因素打分。

当前已纳入：VRS 基础实力、HLTV Rating、火力代理、地图样本深度、样本置信度、赛区对手池代理、阵容数据置信度、种子路径。BO1/BO3、地图 ban/pick、V社排名分层地图胜率和比分质量的影响在主模型预测报告 `reports/iem_cologne_2026_stage1_prediction_vrs-tier-scoreline-map-catboost_2026-05-25.md` 中由瑞士轮模拟器处理。


In [ ]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'stage1_predictor').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

factor_snapshot = ROOT / 'data/snapshots/iem_cologne_major_2026_stage1_vrs_tier_scoreline_features_2026-05-25.csv'
prediction_json = ROOT / 'reports/iem_cologne_2026_stage1_prediction_vrs-tier-scoreline-map-catboost_2026-05-25.json'
radar_svg = ROOT / 'reports/figures/iem_cologne_2026_stage1_vrs_tier_scoreline_radar_2026-05-25.svg'
team_png_dir = ROOT / 'reports/figures/teams'

with factor_snapshot.open(newline='', encoding='utf-8') as handle:
    factor_rows = list(csv.DictReader(handle))

prediction = json.loads(prediction_json.read_text(encoding='utf-8'))
len(factor_rows), prediction['pickem_card']


## 雷达图维度

- 基础实力：VRS 积分百分位。
- HLTV状态：HLTV Rating 3.0 百分位。
- 火力：Rating、K/D、每图 K-D 差的综合代理。
- 地图样本：HLTV 地图样本量代理，表示地图池数据厚度，不等同于强图。
- 样本置信：样本量惩罚后的置信度。
- 对手池：赛区代理，先处理跨赛区强度偏差。
- 阵容数据：全队历史统计与当前阵容页代理的可信度差异。
- 种子路径：Stage 1 种子对瑞士轮路径的影响。

暂未纳入雷达图打分的因素包括：完整 ban/pick 倾向、单地图 vs Top 5/10/20/30 胜率、T/CT 阵营、经济局、选手角色、风格克制、韧性、同阵容交手和旅行/健康等外部因素。它们会继续作为下一阶段数据抓取目标。


In [ ]:
from stage1_predictor.radar_chart import RADAR_DIMENSIONS, load_predictions, render_radar_grid, render_team_pngs

svg = render_radar_grid(factor_rows, columns=4)
radar_svg.parent.mkdir(parents=True, exist_ok=True)
radar_svg.write_text(svg, encoding='utf-8')
render_team_pngs(factor_rows, team_png_dir, load_predictions(prediction_json), 'vrs-tier-scoreline-map-catboost')
radar_svg, team_png_dir


In [ ]:
try:
    from IPython.display import SVG, display
    display(SVG(filename=str(radar_svg)))
except Exception:
    print(radar_svg.read_text(encoding='utf-8')[:1000])


In [ ]:
prob_by_team = {row['team']: row for row in prediction['probabilities']}
ranked = sorted(factor_rows, key=lambda row: float(row['factor_score']), reverse=True)
summary = []
for index, row in enumerate(ranked, start=1):
    item = {
        '排名': index,
        '队伍': row['team'],
        '因素分': row['factor_score'],
        '综合百分位': row['overall_factor_rating'],
        '主模型晋级概率': prob_by_team.get(row['team'], {}).get('p_advance', ''),
    }
    for field, label in RADAR_DIMENSIONS:
        item[label] = row[field]
    summary.append(item)
summary
